In [ ]:
import pandas as pd
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
importlib.reload(gpt)

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables'

In [ ]:
in_df = pd.read_excel(os.path.join(DIR, 'SR_files_with_pubmed_title_abstract_20250609.xlsx'))

In [ ]:
SYSTEM_PROMPT = '''
You are a scientist. You are given a set of inclusion and exclusion criteria and a list of articles. You must decide
whether to include each article and explain why. 
Try your best even if the abstract is missing.

The format should be as follows:

```
## ARTICLE 1

<reasoning>

INCLUDE or EXCLUDE
```

You must make a decision for **each** input and write it only once.
'''
CRITERIA = '''
## INCLUSION CRITERIA

- Study type: Observational studies published in peer-reviewed literature.
- Participants: Individuals tested for SARS-CoV-2 by serological or molecular tests, with or without control groups.
- Outcomes: Studies reporting physical and/or mental health symptoms, symptom clusters, new onset symptoms, or ongoing symptoms following initial SARS-CoV-2 infection.
- Timeframe: No restriction on the time elapsed between infection and symptom reporting, as long as participants had a prior SARS-CoV-2 infection.
- Language: Only studies published in English.
- Publication date: Studies published between January 2020 and August 2022.
- Geographic location: No restrictions applied.
'''


In [ ]:
len(in_df)

In [ ]:
out_df.decision.sum(), out_df.decision.mean()

In [ ]:
out_df.to_excel(os.path.join(DIR, 'SR_files_with_gpt_decision.xlsx'), index=False)

## Fixing

In [ ]:
bad_df = out_df[out_df.decision.isna()]
len(bad_df)

In [ ]:
bad_df = gpt._validate_columns(bad_df)
batches = list(batched(bad_df.itertuples(), 50))

In [ ]:
# response = gpt.query(model=gpt.Models.MINI, instructions=SYSTEM_PROMPT, input=gpt._make_prompt(batches[2], CRITERIA))
# print(response)

In [ ]:
parsed = gpt._parse_eligibility_output(response, 50)
parsed

In [ ]:
importlib.reload(gpt)

In [ ]:
gpt.fill_in_eligibility(
    out_df,
    criteria=CRITERIA)

In [ ]:
out_df.to_excel(os.path.join(DIR, 'SR_files_with_gpt_decision_2025-06-10.xlsx'), index=False)